# 07 - Previsão de nota baixa antes da compra

**Pergunta:** é possível identificar, no momento da compra, pedidos com alta chance de nota baixa?

Variáveis usadas são apenas as conhecidas antes da entrega: prazo estimado, valores, frete, categoria, estado, pagamento e histórico do vendedor (calculado só com dados de treino).

**Critério de decisão (definido antes de rodar):** alerta viável se, entre os 10% de maior risco, a taxa de nota baixa for ≥ 25% e a precisão ≥ 25%.

## 1. Configuração e base

Mesma base do notebook 06: pedidos entregues com nota, no período de jan/2017 a ago/2018.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import (roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)

PROCESSED = Path("../data/processed")
RAW = Path("../data/raw")
TAB = Path("../reports/tabelas")
TAB.mkdir(parents=True, exist_ok=True)

base = pd.read_csv(PROCESSED / "base_pedidos.csv",
                   parse_dates=["order_purchase_timestamp", "order_estimated_delivery_date"])
base["mes_compra"] = base["order_purchase_timestamp"].dt.to_period("M")

from utils import JANELA_INICIO as INICIO, JANELA_FIM as FIM, SEED
janela = base[(base["mes_compra"] >= INICIO) & (base["mes_compra"] <= FIM)]

df = janela[
    (janela["order_status"] == "delivered")
    & janela["review_score"].notna()
    & ~janela["excluir_logistica"]
].copy()
df["nota_baixa"] = (df["review_score"] <= 2).astype(int)

print(f"Pedidos entregues com nota: {len(df):,}")
print(f"Taxa de nota baixa: {df['nota_baixa'].mean():.1%}")

Pedidos entregues com nota: 94,198
Taxa de nota baixa: 12.8%


## 2. Variáveis conhecidas antes da entrega

Categoria e vendedor vêm do primeiro item do pedido. Prazo e frete são calculados a partir de campos disponíveis no momento da compra.

In [2]:
itens = pd.read_csv(RAW / "olist_order_items_dataset.csv",
                    usecols=["order_id", "order_item_id", "product_id", "seller_id"])
prod = pd.read_csv(RAW / "olist_products_dataset.csv", usecols=["product_id", "product_category_name"])
trad = pd.read_csv(RAW / "product_category_name_translation.csv")

itens = (itens.merge(prod, on="product_id", how="left")
              .merge(trad, on="product_category_name", how="left"))
itens["categoria"] = (itens["product_category_name_english"]
                      .fillna(itens["product_category_name"]).fillna("sem_categoria"))

principal = (itens.sort_values("order_item_id")
                  .drop_duplicates("order_id")[["order_id", "categoria", "seller_id"]])
df = df.merge(principal, on="order_id", how="left")

df["prazo_estimado_dias"] = (
    (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df["frete_pct"] = df["frete"] / df["receita_produtos"].replace(0, np.nan)

top_cat = df["categoria"].value_counts().head(15).index
df["categoria_principal"] = df["categoria"].where(df["categoria"].isin(top_cat), "outras")

NUM = ["prazo_estimado_dias", "receita_produtos", "frete", "frete_pct",
       "qtd_itens", "qtd_sellers", "parcelas_max", "seller_taxa"]
CAT = ["categoria_principal", "customer_state", "tipo_pagamento"]

print(df[["prazo_estimado_dias", "frete_pct", "categoria_principal"]].describe(include="all").round(2))

        prazo_estimado_dias  frete_pct categoria_principal
count              94198.00   94198.00               94198
unique                  NaN        NaN                  16
top                     NaN        NaN              outras
freq                    NaN        NaN               19926
mean                  23.65       0.31                 NaN
std                    8.61       0.31                 NaN
min                    2.01       0.00                 NaN
25%                   18.33       0.13                 NaN
50%                   23.22       0.22                 NaN
75%                   28.38       0.38                 NaN
max                  155.14      21.45                 NaN


## 3. Divisão temporal e histórico do vendedor

A divisão é por data, para simular uso real: treino antes de 2018, validação em jan–mar de 2018 e teste de abril de 2018 em diante.

A taxa de nota baixa por vendedor é suavizada para a média geral e calculada só com dados de treino. No treino, usa-se validação cruzada (out-of-fold): cada linha é codificada sem usar o próprio rótulo, o que evita vazamento.

In [3]:
CORTE_VAL = pd.Timestamp("2018-01-01")
CORTE_TESTE = pd.Timestamp("2018-04-01")
ts = df["order_purchase_timestamp"]

treino = df[ts < CORTE_VAL].copy()
valid = df[(ts >= CORTE_VAL) & (ts < CORTE_TESTE)].copy()
teste = df[ts >= CORTE_TESTE].copy()

def taxa_vendedor(fit, aplicar, k=20):
    prior = fit["nota_baixa"].mean()
    g = fit.groupby("seller_id")["nota_baixa"].agg(["sum", "count"])
    enc = (g["sum"] + k * prior) / (g["count"] + k)
    return aplicar["seller_id"].map(enc).fillna(prior).values

def taxa_vendedor_oof(d, k=20):
    out = np.zeros(len(d))
    for i_tr, i_va in KFold(5, shuffle=True, random_state=SEED).split(d):
        out[i_va] = taxa_vendedor(d.iloc[i_tr], d.iloc[i_va], k)
    return out

treino["seller_taxa"] = taxa_vendedor_oof(treino)
valid["seller_taxa"] = taxa_vendedor(treino, valid)
teste["seller_taxa"] = taxa_vendedor(treino, teste)

print(f"Treino: {len(treino):,} | Validação: {len(valid):,} | Teste: {len(teste):,}")
print(f"Taxa de nota baixa: treino {treino['nota_baixa'].mean():.1%} | "
      f"validação {valid['nota_baixa'].mean():.1%} | teste {teste['nota_baixa'].mean():.1%}")

Treino: 42,996 | Validação: 20,430 | Teste: 30,772
Taxa de nota baixa: treino 12.1% | validação 18.0% | teste 10.4%


## 4. Pré-processamento e modelos

Comparação entre baseline (sempre nota OK), regressão logística e gradient boosting, ambos com pesos balanceados por classe. O limiar é escolhido na validação, nunca no teste.

In [4]:
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
])

modelos = {
    "Regressão logística balanceada": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED),
    "Gradient boosting balanceado": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
}

X_tr, y_tr = treino[NUM + CAT], treino["nota_baixa"]
X_va, y_va = valid[NUM + CAT], valid["nota_baixa"]
X_te, y_te = teste[NUM + CAT], teste["nota_baixa"]

def limiar_f1(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return thr[np.argmax(f1[:-1])]

def taxa_top10(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    topo = np.argsort(-p)[: int(len(p) * 0.10)]
    return y[topo].mean()

taxa_base = y_te.mean()
resultados = []

base_pred = np.zeros(len(y_te), dtype=int)
resultados.append({
    "modelo": "Baseline: sempre nota OK", "limiar": np.nan,
    "ROC-AUC": 0.5, "acurácia": (base_pred == y_te).mean(),
    "precisão": np.nan, "recall": 0.0, "F1": 0.0,
    "nota baixa no top 10%": taxa_base,
})

for nome, m in modelos.items():
    pipe = Pipeline([("pre", pre), ("m", m)]).fit(X_tr, y_tr)
    p_va = pipe.predict_proba(X_va)[:, 1]
    lim = limiar_f1(y_va, p_va)

    p_te = pipe.predict_proba(X_te)[:, 1]
    pred = (p_te >= lim).astype(int)

    resultados.append({
        "modelo": nome, "limiar": round(lim, 3),
        "ROC-AUC": roc_auc_score(y_te, p_te),
        "acurácia": (pred == y_te).mean(),
        "precisão": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "nota baixa no top 10%": taxa_top10(y_te, p_te),
    })

tab = pd.DataFrame(resultados).round(3)
tab.to_csv(TAB / "modelo_preditivo_metricas.csv", index=False)

print(f"Taxa de nota baixa no teste: {taxa_base:.1%}\n")
print(tab.to_string(index=False))

Taxa de nota baixa no teste: 10.4%

                        modelo  limiar  ROC-AUC  acurácia  precisão  recall    F1  nota baixa no top 10%
      Baseline: sempre nota OK     NaN    0.500     0.896       NaN   0.000 0.000                  0.104
Regressão logística balanceada   0.480    0.617     0.628     0.145   0.525 0.227                  0.234
  Gradient boosting balanceado   0.507    0.610     0.673     0.149   0.455 0.225                  0.243


## 5. Leitura dos resultados

**Critério de decisão:** alerta viável se "nota baixa no top 10%" ≥ 25% e precisão ≥ 25%.

- Se o critério for atingido, o relatório pode propor um alerta no momento da compra, com a ressalva de que o modelo mede risco, não causa.
- Se não for atingido, o relatório mantém o diagnóstico e registra que a previsão antecipada não se mostrou viável com as variáveis disponíveis.